# 1. Handle Outliers

This notebook detects and handles outliers in the continuous numerical columns of the credit card fraud dataset using log1p transformations for heavily skewed features (`amount`, `velocity_last_24h`, `city_population`) and 3-Sigma capping for other continuous features.

### 1. Import Dependencies

In [38]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
print("Setup complete.")

Setup complete.


### 2. Load Processed Data

In [39]:
df = pd.read_csv('../../dataset/processed/credit_card_fraud_null_handled.csv')
print(f"Loaded dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

Loaded dataset: 1,296,675 rows x 13 columns


,distance_to_merchant,customer_age,transaction_hour,day_of_week,is_weekend,location_mismatch,velocity_last_24h,foreign_transaction,amount,merchant_category,city_population,gender,is_fraud
0,127.606239,32,12,1,0,1,0,0,7.27,misc_net,1645,F,0
1,110.308921,32,8,2,0,1,1,0,52.94,gas_transport,1645,F,0
2,21.787261,32,8,2,0,0,2,0,82.08,gas_transport,1645,F,0
3,87.204215,32,12,2,0,1,3,0,34.79,kids_pets,1645,F,0
4,74.212965,32,13,2,0,0,3,0,27.18,home,1645,F,0


### 3. Numerical Columns Inspection & Box Plots

In [40]:
numerical_columns = ['amount', 'velocity_last_24h', 'customer_age', 'distance_to_merchant', 'city_population']
print("Numerical columns to inspect:", numerical_columns)
df[numerical_columns].describe()

Numerical columns to inspect: ['amount', 'velocity_last_24h', 'customer_age', 'distance_to_merchant', 'city_population']


,amount,velocity_last_24h,customer_age,distance_to_merchant,city_population
count,1.296675e+06,1.296675e+06,1.296675e+06,1.296675e+06,1.296675e+06
mean,7.035104e+01,3.882392e+00,4.552822e+01,7.611465e+01,8.882444e+04
std,1.603160e+02,3.081161e+00,1.740895e+01,2.911693e+01,3.019564e+05
min,1.000000e+00,0.000000e+00,1.300000e+01,2.225452e-02,2.300000e+01
25%,9.650000e+00,2.000000e+00,3.200000e+01,5.533491e+01,7.430000e+02
50%,4.752000e+01,3.000000e+00,4.400000e+01,7.823175e+01,2.456000e+03
75%,8.314000e+01,5.000000e+00,5.700000e+01,9.850327e+01,2.032800e+04
max,2.894890e+04,3.500000e+01,9.500000e+01,1.521172e+02,2.906700e+06


In [41]:
# Create box plots for raw features visualization
fig, axes = plt.subplots(1, len(numerical_columns), figsize=(16, 5))
for idx, col in enumerate(numerical_columns):
    sns.boxplot(y=df[col], ax=axes[idx], color='skyblue')
    axes[idx].set_title(f'Raw: Boxplot of {col}')
plt.tight_layout()
plt.show()

C:\Users\2021ICTS28\AppData\Local\Temp\ipykernel_20588\119066006.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 4. Log Transformations for Skewed Columns

Apply log1p transformations to `amount`, `velocity_last_24h`, and `city_population` to compress the extreme right skew.

In [42]:
log_columns = ['amount', 'velocity_last_24h', 'city_population']
for col in log_columns:
    df[f'{col}_log'] = np.log1p(df[col])
print("Log transformations completed. Columns added:", [f'{col}_log' for col in log_columns])

Log transformations completed. Columns added: ['amount_log', 'velocity_last_24h_log', 'city_population_log']


In [43]:
# Create box plots for log-transformed features visualization
fig, axes = plt.subplots(1, len(log_columns), figsize=(12, 5))
for idx, col in enumerate(log_columns):
    sns.boxplot(y=df[f'{col}_log'], ax=axes[idx], color='salmon')
    axes[idx].set_title(f'Log: Boxplot of {col}_log')
plt.tight_layout()
plt.show()

C:\Users\2021ICTS28\AppData\Local\Temp\ipykernel_20588\405740275.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 5. Outlier Capping (Winsorization) for Untransformed Columns

Apply 3-Sigma/empirical rule outlier capping for `customer_age` and `distance_to_merchant` if needed.

In [44]:
cap_columns = ['customer_age', 'distance_to_merchant']
for col in cap_columns:
    mean_val = df[col].mean()
    std_val = df[col].std()
    lower_fence = mean_val - 3 * std_val
    upper_fence = mean_val + 3 * std_val
    
    outliers_lower = (df[col] < lower_fence).sum()
    outliers_upper = (df[col] > upper_fence).sum()
    print(f"Column '{col}': mean = {mean_val:.2f}, std = {std_val:.2f}, bounds = [{lower_fence:.2f}, {upper_fence:.2f}]")
    print(f"  Outliers below 3-sigma: {outliers_lower} ({outliers_lower/len(df)*100:.2f}%)")
    print(f"  Outliers above 3-sigma: {outliers_upper} ({outliers_upper/len(df)*100:.2f}%)")
    
    df[col] = df[col].clip(lower=lower_fence, upper=upper_fence)

Column 'customer_age': mean = 45.53, std = 17.41, bounds = [-6.70, 97.76]
  Outliers below 3-sigma: 0 (0.00%)
  Outliers above 3-sigma: 0 (0.00%)
Column 'distance_to_merchant': mean = 76.11, std = 29.12, bounds = [-11.24, 163.47]
  Outliers below 3-sigma: 0 (0.00%)
  Outliers above 3-sigma: 0 (0.00%)


### 6. Save Processed Dataset

In [45]:
output_path = '../../dataset/processed/credit_card_fraud_outliers_handled.csv'
df.to_csv(output_path, index=False)
print(f"Saved outlier-handled dataset to: {output_path}")

Saved outlier-handled dataset to: ../../dataset/processed/credit_card_fraud_outliers_handled.csv
